# Local Mode and Development

During development, you often want to test your flows without a SLURM cluster.
`prefect-submitit` provides a local execution mode that runs tasks in subprocesses
on your machine, using the same code paths and serialization as SLURM mode.

You'll learn how to:
- Enable local mode via constructor argument or environment variable
- Propagate environment variables to compute workers
- Capture and inspect task logs
- Switch between local and SLURM for your development workflow

## Setup

Make sure the Prefect server is running before executing this notebook:
```bash
pixi run prefect-start
```

In [ ]:
import os

from prefect import flow, task
from prefect_submitit import SlurmTaskRunner

## Define Tasks

In [ ]:
@task
def get_env_var(name: str) -> str | None:
    """Read an environment variable from the worker process."""
    return os.environ.get(name)


@task
def print_marker(marker: str) -> str:
    """Print a marker string (captured in logs) and return it."""
    print(f"MARKER: {marker}")
    return marker


@task
def add(a: int, b: int) -> int:
    return a + b

## Local Mode via Constructor

Pass `execution_mode="local"` to `SlurmTaskRunner`. SLURM-specific parameters
(partition, mem_gb, etc.) are accepted but ignored with a warning.

In [ ]:
local_runner = SlurmTaskRunner(execution_mode="local")


@flow(task_runner=local_runner)
def local_flow():
    future = add.submit(10, 20)
    result = future.result()
    print(f"10 + 20 = {result}")
    return result


assert local_flow() == 30

## Local Mode via Environment Variable

Set `SLURM_TASKRUNNER_BACKEND=local` to use local mode without changing your code.
This is useful for CI/CD pipelines or development machines without SLURM.

In [ ]:
# Set the env var before creating the runner
os.environ["SLURM_TASKRUNNER_BACKEND"] = "local"

# No execution_mode specified — it reads from the environment
env_runner = SlurmTaskRunner()


@flow(task_runner=env_runner)
def env_local_flow():
    future = add.submit(5, 7)
    return future.result()


assert env_local_flow() == 12
print("Running in local mode via SLURM_TASKRUNNER_BACKEND env var")

# Clean up
del os.environ["SLURM_TASKRUNNER_BACKEND"]

## Environment Variable Propagation

Environment variables set on the submit node are automatically available
on compute workers (both local and SLURM).

In [ ]:
# Set a custom env var before submitting
os.environ["MY_APP_CONFIG"] = "production"


@flow(task_runner=SlurmTaskRunner(execution_mode="local"))
def env_propagation_flow():
    future = get_env_var.submit(name="MY_APP_CONFIG")
    result = future.result()
    print(f"MY_APP_CONFIG on worker: {result}")
    return result


result = env_propagation_flow()
assert result == "production"

# Clean up
del os.environ["MY_APP_CONFIG"]

## Log Capture

Anything your task prints to stdout/stderr is captured and accessible
via `future.logs()`, which returns a `(stdout, stderr)` tuple.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode="local"))
def log_capture_flow():
    future = print_marker.submit(marker="HELLO_FROM_WORKER")
    result = future.result()

    stdout, stderr = future.logs()
    print(f"Task returned: {result}")
    print(f"Captured stdout:\n{stdout}")
    print(f"Captured stderr:\n{stderr}")

    return stdout


stdout = log_capture_flow()
assert "HELLO_FROM_WORKER" in stdout

## Development Workflow: Switching Between Local and SLURM

A common pattern is to develop and test locally, then switch to SLURM for production.
You can control this with a single variable or environment check.

In [ ]:
import shutil


def make_runner(**overrides):
    """Create a runner that auto-detects local vs SLURM."""
    # Use SLURM if available, otherwise fall back to local
    has_slurm = shutil.which("srun") is not None
    mode = "slurm" if has_slurm else "local"

    defaults = {
        "execution_mode": mode,
        "partition": "cpu",
        "mem_gb": 4,
        "time_limit": "01:00:00",
    }
    defaults.update(overrides)
    return SlurmTaskRunner(**defaults)


runner = make_runner()
print(f"Using execution mode: {runner.execution_mode}")


@flow(task_runner=runner)
def portable_flow():
    future = add.submit(100, 200)
    return future.result()


assert portable_flow() == 300